In [ ]:
from pyspark.sql import functions as F, types as T

dbutils.widgets.text('environment', 'dev')
dbutils.widgets.text('catalog_name', 'naturapet_dev')
dbutils.widgets.text('storage_account', 'demodldb')
dbutils.widgets.text('storage_container', 'democodex')
dbutils.widgets.text('data_year', '2026')
dbutils.widgets.text('domain_name', 'comercial')

environment = dbutils.widgets.get('environment').strip().lower()
catalog = dbutils.widgets.get('catalog_name').strip()
storage_account = dbutils.widgets.get('storage_account').strip()
storage_container = dbutils.widgets.get('storage_container').strip()
data_year = dbutils.widgets.get('data_year').strip()
domain = dbutils.widgets.get('domain_name').strip().lower()
silver_schema = f'{domain}_silver'
shared_schema = 'shared_silver'
storage_uri = f'abfss://{storage_container}@{storage_account}.dfs.core.windows.net'
quality_path = f'{storage_uri}/external/{environment}/{domain}/silver/{data_year}/_control/silver_quality_checks'
quality_table = f'`{catalog}`.`{silver_schema}`.`silver_quality_checks`'

spark.sql(f'CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{silver_schema}`')

PRIMARY_KEYS = {
    'dim_fecha': ['fecha_id'], 'dim_ciudad': ['ciudad_id'], 'dim_tienda': ['tienda_id'],
    'dim_area_negocio': ['area_id'], 'dim_centro_costo': ['centro_costo_id'],
    'dim_categoria_producto': ['categoria_id'], 'dim_producto': ['producto_id'],
    'dim_proveedor': ['proveedor_id'], 'bridge_producto_proveedor': ['producto_id', 'proveedor_id'],
    'dim_cliente': ['cliente_id'], 'dim_empleado': ['empleado_id'], 'dim_canal_venta': ['canal_id'],
    'dim_metodo_pago': ['metodo_pago_id'], 'dim_campana_marketing': ['campana_id'],
    'dim_transportista': ['transportista_id'], 'dim_motivo_devolucion': ['motivo_devolucion_id'],
    'fact_ventas_cabecera': ['venta_id'], 'fact_ventas_detalle': ['detalle_venta_id'],
    'fact_devoluciones': ['devolucion_id'], 'fact_marketing': ['marketing_id'],
    'fact_inventario_mensual': ['mes_carga', 'tienda_id', 'producto_id'],
    'fact_ajustes_inventario': ['ajuste_id'], 'fact_compras': ['orden_compra_id'],
    'fact_logistica_entregas': ['entrega_id'], 'fact_costos_mensual': ['costo_id'],
    'fact_presupuesto_mensual': ['presupuesto_id'], 'fact_objetivos_mensuales': ['objetivo_id'],
    'fact_calidad_datos_carga': ['control_id'], 'resumen_mensual_validacion': ['mes_carga']
}
FK_CHECKS = [
    ('dim_tienda', 'ciudad_id', 'dim_ciudad', 'ciudad_id'),
    ('dim_cliente', 'ciudad_id', 'dim_ciudad', 'ciudad_id'),
    ('dim_empleado', 'tienda_id', 'dim_tienda', 'tienda_id'),
    ('dim_empleado', 'area_id', 'dim_area_negocio', 'area_id'),
    ('dim_producto', 'categoria_id', 'dim_categoria_producto', 'categoria_id'),
    ('dim_proveedor', 'ciudad_id', 'dim_ciudad', 'ciudad_id'),
    ('bridge_producto_proveedor', 'producto_id', 'dim_producto', 'producto_id'),
    ('bridge_producto_proveedor', 'proveedor_id', 'dim_proveedor', 'proveedor_id'),
    ('fact_ventas_cabecera', 'fecha_id', 'dim_fecha', 'fecha_id'),
    ('fact_ventas_cabecera', 'tienda_id', 'dim_tienda', 'tienda_id'),
    ('fact_ventas_cabecera', 'cliente_id', 'dim_cliente', 'cliente_id'),
    ('fact_ventas_cabecera', 'canal_id', 'dim_canal_venta', 'canal_id'),
    ('fact_ventas_detalle', 'venta_id', 'fact_ventas_cabecera', 'venta_id'),
    ('fact_ventas_detalle', 'producto_id', 'dim_producto', 'producto_id'),
    ('fact_devoluciones', 'detalle_venta_id', 'fact_ventas_detalle', 'detalle_venta_id'),
    ('fact_logistica_entregas', 'venta_id', 'fact_ventas_cabecera', 'venta_id'),
    ('fact_compras', 'producto_id', 'dim_producto', 'producto_id'),
    ('fact_compras', 'proveedor_id', 'dim_proveedor', 'proveedor_id'),
    ('fact_inventario_mensual', 'producto_id', 'dim_producto', 'producto_id')
]

def table_exists(schema_name, table_name):
    return spark.catalog.tableExists(f'{catalog}.{schema_name}.{table_name}')

def resolve_table(table_name):
    if table_exists(silver_schema, table_name):
        return f'`{catalog}`.`{silver_schema}`.`{table_name}`'
    if table_exists(shared_schema, table_name):
        return f'`{catalog}`.`{shared_schema}`.`{table_name}`'
    return None

def add_check(checks, table_name, check_name, severity, failing_count, description):
    checks.append({
        'environment': environment,
        'domain_name': domain,
        'table_name': table_name,
        'check_name': check_name,
        'severity': severity,
        'failing_count': int(failing_count),
        'status': 'PASS' if int(failing_count) == 0 else 'FAIL',
        'description': description
    })

checks = []
tables = [
    row.tableName
    for row in spark.sql(f'SHOW TABLES IN `{catalog}`.`{silver_schema}`').collect()
    if not row.isTemporary and row.tableName not in {'silver_quality_checks', 'silver_processing_audit'}
]

if not tables:
    add_check(checks, '__NO_TABLES__', 'no_silver_tables', 'INFO', 0, 'No existen tablas Silver de negocio para validar en este dominio')

for table_name in tables:
    df = spark.table(f'`{catalog}`.`{silver_schema}`.`{table_name}`')
    keys = [key for key in PRIMARY_KEYS.get(table_name, []) if key in df.columns]
    if keys:
        duplicates = df.groupBy(*keys).count().filter(F.col('count') > 1).count()
        add_check(checks, table_name, 'primary_key_unique', 'ERROR', duplicates, f'PK unica por {keys}')
        null_keys = df.filter(' OR '.join([f'`{key}` IS NULL' for key in keys])).count()
        add_check(checks, table_name, 'primary_key_not_null', 'ERROR', null_keys, f'PK no nula por {keys}')
    add_check(checks, table_name, 'table_not_empty', 'ERROR', 0 if df.limit(1).count() else 1, 'La tabla Silver debe tener registros')

for source_table, source_col, target_table, target_col in FK_CHECKS:
    source_ref = resolve_table(source_table)
    target_ref = resolve_table(target_table)
    if not source_ref or not target_ref:
        continue
    source_df = spark.table(source_ref)
    target_df = spark.table(target_ref)
    if source_col not in source_df.columns or target_col not in target_df.columns:
        continue
    failures = (source_df.filter(F.col(source_col).isNotNull())
                .select(F.col(source_col).alias('_fk')).distinct()
                .join(target_df.select(F.col(target_col).alias('_pk')).distinct(), F.col('_fk') == F.col('_pk'), 'left_anti')
                .count())
    add_check(checks, source_table, f'fk_{source_col}_to_{target_table}_{target_col}', 'ERROR', failures, 'Integridad referencial Silver')

def arithmetic_check(table_name, condition, description):
    if not table_exists(silver_schema, table_name):
        return
    failures = spark.table(f'`{catalog}`.`{silver_schema}`.`{table_name}`').filter(condition).count()
    add_check(checks, table_name, 'arithmetic_consistency', 'ERROR', failures, description)

arithmetic_check('fact_ventas_cabecera', 'abs(cast(valor_bruto - descuento_total + iva_total - valor_total as double)) > 0.02', 'valor_total = valor_bruto - descuento_total + iva_total')
arithmetic_check('fact_ventas_cabecera', 'abs(cast(valor_bruto - descuento_total - costo_total - margen_bruto as double)) > 0.02', 'margen_bruto = base_neta_sin_iva - costo_total')
arithmetic_check('fact_ventas_detalle', 'abs(cast(base_sin_iva + iva_valor - valor_total_linea as double)) > 0.02', 'valor_total_linea = base_sin_iva + iva_valor')
arithmetic_check('fact_ventas_detalle', 'abs(cast(base_sin_iva - costo_total_linea - margen_bruto_linea as double)) > 0.02', 'margen_bruto_linea = base_sin_iva - costo_total_linea')
arithmetic_check('fact_compras', 'abs(cast(subtotal_sin_iva + iva_valor - valor_total_compra as double)) > 0.02', 'valor_total_compra = subtotal_sin_iva + iva_valor')
arithmetic_check('fact_inventario_mensual', 'stock_inicial + entradas_compras - salidas_ventas + ajuste_unidades - merma_unidades <> stock_final', 'stock_final cuadra con movimientos')

schema = T.StructType([
    T.StructField('environment', T.StringType(), False),
    T.StructField('domain_name', T.StringType(), False),
    T.StructField('table_name', T.StringType(), False),
    T.StructField('check_name', T.StringType(), False),
    T.StructField('severity', T.StringType(), False),
    T.StructField('failing_count', T.LongType(), False),
    T.StructField('status', T.StringType(), False),
    T.StructField('description', T.StringType(), False)
])
quality_df = spark.createDataFrame(checks, schema=schema).withColumn('checked_at', F.current_timestamp())
(quality_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').option('path', quality_path).saveAsTable(quality_table))
display(quality_df.orderBy(F.desc('status'), 'table_name', 'check_name'))
